# Research foundations: one connected historical-data workflow

**Question:** can we go from the selected Binance job to actual saved trades and exact financial values using the installed package?

Follow the same real inputs through **configure → acquire/resume → verify → select the hour → inspect clocks → calculate exact values → verify repeat/failures**. Here “real” means the historical Binance archive, not a streaming feed. Each step explains its inputs, work and output. No synthetic success path is required.

The fixed case remains BTCUSDT, `[2024-06-01 00:00, 01:00)` UTC. We download its containing day and keep raw data under ignored `data/binance-archives/`. An ordinary **Run All** fetches only when this run is incomplete; a repeat verifies and reuses the saved bytes. No API key is needed. No trading or strategy backtest runs here.

The [companion notebook](foundation-contract-checks.ipynb) retains the detailed synthetic clock, decoder, hand-ledger and process-failure checkpoints. Use it when you want to understand a boundary that this real sample cannot demonstrate. [Run commands and environment setup](README.md).

## 1. Check the installed environment

**Input:** the selected Python kernel and this checkout. **Work:** verify the pinned interpreter and compare every runtime Python/JSON/type-marker file with the installed package. **Output:** a small identity table tying this run to exact code, notebook source and lockfile.

This check catches a stale notebook kernel or install before interpreting data. It does not modify `sys.path`. Use the repository kernel and restart it after installing changed code.

In [1]:
# standard library imports
import hashlib
import json
import os
import platform
import subprocess
import sys
from datetime import UTC, datetime, timedelta
from decimal import Decimal
from html import escape
from importlib.metadata import distributions
from pathlib import Path

# third party library imports
from IPython.display import HTML, display

# project library imports
import alpha_research
from alpha_research.financial_units import (
    Dimension,
    Money,
    Price,
    Quantity,
    QuantityKind,
)
from alpha_research.reference_data import CatalogSnapshot
from alpha_research.serialization import DomainCodec

root = Path.cwd().resolve()
if root.name == "examples":
    root = root.parent
assert (root / "configs/reference/binance-archive-spot.json").is_file(), (
    "Run from the repository root or examples/."
)
notebook_path = Path(
    os.environ.get("ALPHA_RESEARCH_NOTEBOOK_PATH", root / "examples/research-foundations.ipynb")
).resolve()


class NotebookDisplay:
    """Render small tables for the exploratory notebook scenarios."""

    @staticmethod
    def show_table(headers: tuple[str, ...], rows: list[tuple[object, ...]]) -> None:
        """Display a small escaped table without adding a dataframe dependency.

        Args:
            headers: Column labels.
            rows: Exact or presentation values to render as text.
        """
        heading = "".join(
            f'<th style="text-align:left;padding:6px">{escape(label)}</th>' for label in headers
        )
        body = "".join(
            "<tr>"
            + "".join(
                f'<td style="padding:6px;vertical-align:top">{escape(str(value))}</td>'
                for value in row
            )
            + "</tr>"
            for row in rows
        )
        display(HTML(f"<table><thead><tr>{heading}</tr></thead><tbody>{body}</tbody></table>"))

In [2]:
assert platform.python_version() == (root / ".python-version").read_text().strip() == "3.14.7"
installed = Path(alpha_research.__file__).resolve().parent
assert "site-packages" in installed.parts and installed != root / "src/alpha_research"
source_root = root / "src/alpha_research"
runtime_files = sorted(
    path
    for path in source_root.rglob("*")
    if path.is_file() and path.suffix in {".py", ".json", ".typed"}
)
assert runtime_files
runtime_digest = hashlib.sha256()
for source_file in runtime_files:
    relative = source_file.relative_to(source_root)
    source_bytes = source_file.read_bytes()
    assert source_bytes == (installed / relative).read_bytes(), f"Stale installed file: {relative}"
    runtime_digest.update(str(relative).encode() + b"\0" + source_bytes + b"\0")

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=root, text=True).strip()
branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=root, text=True).strip()
working_status = subprocess.check_output(["git", "status", "--short"], cwd=root, text=True).strip()
candidate_diff = subprocess.check_output(
    [
        "git",
        "diff",
        "--binary",
        "HEAD",
        "--",
        "src",
        "configs",
        "pyproject.toml",
        "uv.lock",
        ".python-version",
    ],
    cwd=root,
)
notebook_document = json.loads(notebook_path.read_text())
notebook_source = [
    (cell["cell_type"], "".join(cell["source"])) for cell in notebook_document["cells"]
]
notebook_digest = hashlib.sha256(
    json.dumps(notebook_source, ensure_ascii=False, separators=(",", ":")).encode()
).hexdigest()
identity = {
    "notebook_path": str(notebook_path),
    "attempt_id": os.environ.get("ALPHA_RESEARCH_ATTEMPT_ID", "interactive: unsnapshotted"),
    "commit": commit,
    "branch": branch,
    "working_tree": "changed" if working_status else "clean",
    "python": platform.python_version(),
    "platform": platform.platform(),
    "kernel": sys.executable,
    "package": str(installed),
    "verified_runtime_files": len(runtime_files),
    "runtime_sha256": runtime_digest.hexdigest(),
    "runtime_input_diff_sha256": hashlib.sha256(candidate_diff).hexdigest(),
    "uv_lock_sha256": hashlib.sha256((root / "uv.lock").read_bytes()).hexdigest(),
    "catalog_file_sha256": hashlib.sha256(
        (root / "configs/reference/binance-archive-spot.json").read_bytes()
    ).hexdigest(),
    "notebook_source_sha256": notebook_digest,
    "notebook_tooling": {
        item.metadata["Name"]: item.version
        for item in distributions()
        if item.metadata["Name"] in {"nbclient", "nbformat", "ipykernel"}
    },
}
NotebookDisplay.show_table(("Identity", "Observed value"), list(identity.items()))

Identity,Observed value
notebook_path,/Users/lxg/Documents/GitHub/Alpha Research/alpha-research-lab-private/examples/research-foundations.ipynb
attempt_id,ee11fc43-8a05-4515-bcf9-32519f84675e
commit,a5d86a91e94b70b0e78deff44031834ceef0a294
branch,codex/mr-08-saved-archive-rows
working_tree,changed
python,3.14.7
platform,macOS-27.0-arm64-arm-64bit-Mach-O
kernel,/Users/lxg/.cache/uv/builds-v0/.tmpBsQkSt/bin/python
package,/Users/lxg/Documents/GitHub/Alpha Research/alpha-research-lab-private/.venv/lib/python3.14/site-packages/alpha_research
verified_runtime_files,59


## 2. Resolve the actual research selection

**Input:** a named job and its archive catalog. **Work:** use `Configuration.resolve` and `SourceRequest.partition` from the installed package. **Output:** the exact instrument, requested hour, downloaded day and immutable run identity.

The catalog supplies canonical identity, quote currency and tick assumptions. Its 2026 authorship is visible; it cannot establish historical 2024 eligibility. Binance packages a whole day, so request bounds differ from research bounds. Downstream cells use these objects directly rather than repeating the symbol or interval.

`RUN_REAL_BINANCE=False` (or `ALPHA_BINANCE_DOWNLOAD=0`) means **read the existing completed cache without network access**. Missing or incomplete cache then fails with a recovery instruction. The companion notebook is the fully synthetic/offline alternative when no actual sample exists.

In [3]:
# project library imports
from alpha_research.configuration import Configuration
from alpha_research.data.manifests import AcquisitionManifest
from alpha_research.data.raw_store import RawStore
from alpha_research.data.requests import SourceRequest
from alpha_research.pipelines.acquisition import main as acquisition_main

RUN_REAL_BINANCE = os.environ.get("ALPHA_BINANCE_DOWNLOAD", "1") == "1"
BINANCE_DIRECTORY = (
    Path(os.environ.get("ALPHA_BINANCE_DIRECTORY", str(root / "data/binance-archives")))
    .expanduser()
    .resolve()
)
BINANCE_MAX_BYTES = 64 * 1024 * 1024
JOB_NAME = "btc-archive"

if BINANCE_DIRECTORY.is_relative_to(root) and not BINANCE_DIRECTORY.is_relative_to(
    (root / "data/binance-archives").resolve()
):
    raise ValueError("Choose ignored data/binance-archives or storage outside the checkout.")
real_codec = DomainCodec()
real_configuration = Configuration.read(root / "configs/data-jobs/binance-archive.json")
real_catalog = real_codec.decode_json(
    (root / "configs/reference/binance-archive-spot.json").read_bytes(), CatalogSnapshot
)
real_run = real_configuration.resolve(JOB_NAME, real_catalog)
real_requests = SourceRequest.partition(real_run)
if len(real_requests) != 1:
    raise ValueError(
        "This notebook inspects one selected instrument/day; select the bounded btc-archive job."
    )
real_request = real_requests[0]
real_instrument = real_request.instrument
real_binance_directory = BINANCE_DIRECTORY / "runs" / real_run.identity
real_store = RawStore(BINANCE_DIRECTORY / "raw")
manifest_path = real_binance_directory / "manifest.json"
NotebookDisplay.show_table(
    ("Selection", "Resolved value"),
    [
        (
            "Instrument / canonical ID",
            f"{real_instrument.symbol} / {real_instrument.instrument_id}",
        ),
        ("Provider / timestamp policy", f"{real_run.job.provider} / {real_run.job.capture_policy}"),
        ("Research hour (half-open)", f"{real_run.job.start} → {real_run.job.end}"),
        ("Source day (half-open)", f"{real_request.start} → {real_request.end}"),
        (
            "Catalog recorded / basis",
            f"{real_catalog.recorded_at} / {real_instrument.metadata_basis}",
        ),
        (
            "Quote currency / price tick",
            f"{real_instrument.quote_currency} / {real_instrument.price_tick}",
        ),
        ("Run identity", real_run.identity),
        (
            "Mode",
            "download/resume allowed" if RUN_REAL_BINANCE else "verified cache only; no network",
        ),
    ],
)

Selection,Resolved value
Instrument / canonical ID,BTCUSDT / 11111111-1111-4111-8111-111111111111
Provider / timestamp policy,binance-archive / exchange-time-only-v1
Research hour (half-open),2024-06-01 00:00:00+00:00 → 2024-06-01 01:00:00+00:00
Source day (half-open),2024-06-01 00:00:00+00:00 → 2024-06-02 00:00:00+00:00
Catalog recorded / basis,2026-09-15 09:44:05+00:00 / research_assumption
Quote currency / price tick,code='USDT' minor_unit=Decimal('1E-8') / 0.01
Run identity,6b564380af76ea0ad0d268f983020cf4d602617d77fe103f3d1886cf5f2ba0cd
Mode,download/resume allowed


## 3. Download, resume or reopen the same saved run

**Input:** the resolved run above. **Work:** save exact inputs before I/O, then call the installed `alpha-acquire` entry function. The provider adapter checks the publisher SHA-256 and full ZIP integrity; the shared pipeline publishes bytes and saves a completion checkpoint. **Output:** one verified manifest and separate attempt log.

A completion is recorded only after durable storage. Interrupt and rerun this cell to resume. Completed runs skip HTTP. The timeout is 10 seconds of socket inactivity, with two retries and separate 64 MiB compressed/expanded caps. Compact stage reports appear below; full JSON remains in the attempt log.

The local working copy has no automatic expiry; keep its inputs, manifest and raw bytes together. Today's receipt/ingestion clocks never become historical event availability. The [source contract](../docs/research/p1-baseline.md) records the founder's source/use selection and retention limits.

In [4]:
# standard library imports
from contextlib import redirect_stderr, redirect_stdout
from typing import TextIO
from uuid import uuid4


class _NotebookArchiveFiles:
    """Provision local acquisition directories and preserve exact configuration on repeat runs."""

    @staticmethod
    def directory(destination: Path) -> None:
        """Create missing private ancestors and synchronize each directory entry."""
        if not destination.parent.is_dir():
            _NotebookArchiveFiles.directory(destination.parent)
        destination.mkdir(mode=0o700, exist_ok=True)
        descriptor = os.open(destination.parent, os.O_RDONLY | os.O_DIRECTORY)
        try:
            os.fsync(descriptor)
        finally:
            os.close(descriptor)

    @staticmethod
    def save(destination: Path, payload: bytes) -> None:
        """Preserve existing inputs and fail if their content differs from this selection."""
        try:
            with destination.open("xb") as output:
                output.write(payload)
                output.flush()
                os.fsync(output.fileno())
        except FileExistsError:
            if destination.read_bytes() != payload:
                raise ValueError(
                    f"Saved input differs; choose a new acquisition directory: {destination.name}"
                )


class _NotebookArchiveLog:
    """Save full command JSON while displaying compact progress at each completed line."""

    def __init__(self, saved: TextIO, display: TextIO) -> None:
        """Bind the full attempt log and the original notebook output stream."""
        self._saved = saved
        self._display = display
        self._buffer = ""

    def write(self, value: str) -> int:
        """Save a text chunk and display complete JSON progress records."""
        self._saved.write(value)
        self._saved.flush()
        self._buffer += value
        while "\n" in self._buffer:
            line, self._buffer = self._buffer.split("\n", 1)
            if line:
                record = json.loads(line)
                print(
                    f"{record['stage']}: {record['status']} | "
                    f"completed {record['completed']}/{record['total']} | "
                    f"bytes {record['persisted_bytes']} | retries {record['retries']}",
                    file=self._display,
                    flush=True,
                )
                if "diagnostic" in record:
                    print(record["diagnostic"], file=self._display, flush=True)
        return len(value)

    def flush(self) -> None:
        """Flush the saved attempt log without completing a partial JSON line."""
        self._saved.flush()

In [5]:
if RUN_REAL_BINANCE:
    for directory in (
        BINANCE_DIRECTORY,
        BINANCE_DIRECTORY / "raw",
        BINANCE_DIRECTORY / "runs",
        real_binance_directory,
    ):
        _NotebookArchiveFiles.directory(directory)
    for filename, value in (
        ("jobs.json", real_configuration),
        ("catalog.json", real_catalog),
        (
            "use-and-retention.json",
            {
                "rights_reference": real_run.job.rights_reference,
                "source_reference": "https://github.com/binance/binance-public-data",
                "reviewed_on": "2026-09-15",
                "intended_use": "private offline engineering/research",
                "retention": (
                    "User-selected local working copy; no automatic expiry; "
                    "long-term archival rights not established."
                ),
            },
        ),
    ):
        _NotebookArchiveFiles.save(real_binance_directory / filename, real_codec.encode_json(value))
    _NotebookArchiveFiles.directory(real_binance_directory / "attempts")
    real_binance_arguments = [
        "--config",
        str(real_binance_directory / "jobs.json"),
        "--catalog",
        str(real_binance_directory / "catalog.json"),
        "--job",
        real_run.job.name,
        "--manifest",
        str(manifest_path),
        "--store",
        str(BINANCE_DIRECTORY / "raw"),
        "--rights-reference",
        real_run.job.rights_reference,
        "--timeout-seconds",
        "10",
        "--max-bytes",
        str(BINANCE_MAX_BYTES),
        "--retry-delays",
        "1",
        "2",
    ]
    attempt_path = real_binance_directory / "attempts" / f"{uuid4()}.jsonl"
    with attempt_path.open("x", encoding="utf-8") as attempt:
        progress = _NotebookArchiveLog(attempt, sys.stdout)
        with redirect_stdout(progress), redirect_stderr(progress):
            real_binance_exit = acquisition_main(real_binance_arguments)
        os.fsync(attempt.fileno())
    print(f"Saved full command log: {attempt_path}")
    if real_binance_exit != 0:
        raise RuntimeError(
            "Acquisition stopped. Correct the reported cause and rerun this cell to resume; "
            "keep the manifest/raw files."
        )

real_checkpoint = AcquisitionManifest.read(manifest_path, run=real_run, store=real_store)
if len(real_checkpoint.completed) != len(real_requests) or real_checkpoint.pending is not None:
    raise FileNotFoundError(
        "No complete verified cache for this run. Set RUN_REAL_BINANCE=True and rerun "
        "acquisition, or use foundation-contract-checks.ipynb offline."
    )
descriptor = real_checkpoint.completed[0]
assert descriptor.request_identity == real_request.identity
print(
    f"Ready: {len(real_checkpoint.completed)} verified source object, "
    f"{descriptor.size_bytes:,} bytes. All following cells consume this checkpoint."
)

configuration: started | completed None/None | bytes None | retries 0


resume: verifying | completed None/None | bytes None | retries 0


resume: resumed | completed 0/1 | bytes 0 | retries 0


fetch: started | completed 0/1 | bytes 0 | retries 0


store: received | completed 0/1 | bytes 0 | retries 0


store: stored | completed 0/1 | bytes 0 | retries 0


checkpoint: checkpointing | completed 0/1 | bytes 0 | retries 0


checkpoint: checkpointed | completed 1/1 | bytes 7308819 | retries 0


complete: complete | completed 1/1 | bytes 7308819 | retries 0


Saved full command log: /Users/lxg/Documents/GitHub/Alpha Research/alpha-research-lab-private/data/binance-archives/runs/6b564380af76ea0ad0d268f983020cf4d602617d77fe103f3d1886cf5f2ba0cd/attempts/4139b770-d491-486e-901c-000509cfd1a7.jsonl
Ready: 1 verified source object, 7,308,819 bytes. All following cells consume this checkpoint.


## 4. Connect source identity, storage and clocks

**Input:** the completed descriptor and its request. **Work:** bind the downloaded object to the same resolved run and distinguish the clocks it actually supplies. **Output:** the exact byte digest and original receipt/ingestion times used by every later step.

The ZIP digest matches the publisher checksum checked during download. `ts_received` records local acquisition; `ts_ingested` records durable publication. The CSV supplies exchange trade time, but no recorder capture or historical knowledge time. Repeats must preserve these facts. Canonical point-in-time events cannot be produced by inventing a Tardis capture prefix.

In [6]:
NotebookDisplay.show_table(
    ("Provenance", "Value / meaning"),
    [
        ("Run identity", real_checkpoint.run.identity),
        ("Source request", descriptor.request_identity),
        ("Exact ZIP SHA-256", descriptor.sha256),
        ("Stored bytes", descriptor.size_bytes),
        ("Original local receipt", descriptor.ts_received),
        ("Durable local ingestion", descriptor.ts_ingested),
        ("Exchange event time", "CSV time column; inspected next"),
        ("Recorder capture / historical knowledge", "not supplied / unknown"),
    ],
)

Provenance,Value / meaning
Run identity,6b564380af76ea0ad0d268f983020cf4d602617d77fe103f3d1886cf5f2ba0cd
Source request,c48117ff86b3e215ac330e0568c8019242beed75968c9fdd6ada366e82452cd9
Exact ZIP SHA-256,c9c333f58fe33faaa0a34d526c61b13f5648b908c4ad341dd6556e15c5bcc46a
Stored bytes,7308819
Original local receipt,2026-09-16 07:37:35.441268+00:00
Durable local ingestion,2026-09-16 07:37:35.456346+00:00
Exchange event time,CSV time column; inspected next
Recorder capture / historical knowledge,not supplied / unknown


## 5. Follow the saved bytes into the selected hour — NB 08

**Input:** the same acquisition manifest and raw store. **Work:** use the installed `BinanceArchiveRows` to verify and stream the expected ZIP/UTF-8 CSV with exact row bytes, stable zero-based positions, origins and selection bounds. The exploratory notebook then checks integer trade/time fields, requires timestamps inside the source day, and selects the configured half-open hour. **Output:** counts, first/last selected event, and three actual rows with their archive positions.

The reader retains the full day's source order, including equal-time rows; it neither filters nor constructs an event decoder. The notebook's timestamp preview uses integer milliseconds before 2025 and integer microseconds from 2025. Those exploratory conversions are not canonical normalization (MR 09). Recorder coverage remains **unknown** even after every row is read. Empty content is not a disconnect or a quiet-market claim. The synthetic NB 08 cases in section 7 make these distinctions observable.

In [7]:
# project library imports
from alpha_research.data.providers.binance.rows import BinanceArchiveRows

selected_count, archive_count = 0, 0
selected_first, selected_last = None, None
selected_preview = []
epoch = datetime(1970, 1, 1, tzinfo=UTC)
uses_microseconds = real_request.start >= datetime(2025, 1, 1, tzinfo=UTC)
archive_reader = BinanceArchiveRows.from_manifest(real_checkpoint)[0]
for raw_row in archive_reader.read(real_store, max_bytes=BINANCE_MAX_BYTES):
    position, row = raw_row.position, raw_row.columns()
    if (
        len(row) != 7
        or not row[0].isascii()
        or not row[0].isdecimal()
        or not row[4].isascii()
        or not row[4].isdecimal()
    ):
        raise ValueError(f"Unexpected individual-trade CSV schema at archive row {position}")
    timestamp = epoch + timedelta(microseconds=int(row[4]) * (1 if uses_microseconds else 1000))
    if not real_request.start <= timestamp < real_request.end:
        raise ValueError(f"Trade timestamp is outside the archive day at row {position}")
    archive_count += 1
    if real_run.job.start <= timestamp < real_run.job.end:
        selected_count += 1
        selected_first = timestamp if selected_first is None else min(selected_first, timestamp)
        selected_last = timestamp if selected_last is None else max(selected_last, timestamp)
        if len(selected_preview) < 3:
            selected_preview.append((position, timestamp, tuple(row)))
if not selected_count:
    raise ValueError(
        "The selected hour contains no observations; do not pad prices or claim a usable sample."
    )
NotebookDisplay.show_table(
    ("Observed in saved data", "Value"),
    [
        ("Archive rows", archive_count),
        ("Recorder coverage", archive_reader.recorder_coverage),
        ("Selected-hour rows", selected_count),
        ("First selected event", selected_first),
        ("Last selected event", selected_last),
        ("Exchange timestamp unit", "microseconds" if uses_microseconds else "milliseconds"),
    ],
)
NotebookDisplay.show_table(
    (
        "Archive row",
        "Trade ID",
        "Exchange UTC",
        "Price text",
        "Quantity text",
        "Quote quantity text",
    ),
    [
        (position, row[0], timestamp, row[1], row[2], row[3])
        for position, timestamp, row in selected_preview
    ],
)

Observed in saved data,Value
Archive rows,638484
Recorder coverage,unknown
Selected-hour rows,27480
First selected event,2024-06-01 00:00:00.001000+00:00
Last selected event,2024-06-01 00:59:59.997000+00:00
Exchange timestamp unit,milliseconds


Archive row,Trade ID,Exchange UTC,Price text,Quantity text,Quote quantity text
0,3620989757,2024-06-01 00:00:00.001000+00:00,67540.01000000,0.00099000,66.86460990
1,3620989758,2024-06-01 00:00:00.008000+00:00,67540.01000000,0.00071000,47.95340710
2,3620989759,2024-06-01 00:00:00.414000+00:00,67540.01000000,0.00118000,79.69721180


## 6. Use those actual trades with exact financial units

**Input:** the three preview rows above and the same resolved instrument. **Work:** parse decimal text without floats, construct the package's `Price` and `Quantity`, and use `Price.amount` to calculate quote-currency notional. Round-trip each object through `DomainCodec`. **Output:** exact notionals compared with the archive's quote-quantity column, plus one serialized amount.

The dimension ties quantity to the canonical listing and base unit; the currency comes from the catalog. This connects raw research inspection to the existing financial library without inventing canonical event provenance. The comparison reports any source rounding difference; it is not an accounting ledger, a simulated fill, or validation of every row.

In [8]:
real_dimension = Dimension(instrument_id=real_instrument.instrument_id, kind=QuantityKind.BASE_UNIT)
unit_rows = []
for position, timestamp, row in selected_preview:
    price_value, quantity_value, source_quote = (Decimal(row[index]) for index in (1, 2, 3))
    if not all(
        value.is_finite() and value > 0 for value in (price_value, quantity_value, source_quote)
    ):
        raise ValueError(f"Preview row {position} has an invalid trade amount")
    price = Price(value=price_value, currency=real_instrument.quote_currency, per=real_dimension)
    quantity = Quantity(value=quantity_value, dimension=real_dimension)
    notional = price.amount(quantity)
    assert real_codec.decode_json(real_codec.encode_json(price), Price) == price
    assert real_codec.decode_json(real_codec.encode_json(quantity), Quantity) == quantity
    assert real_codec.decode_json(real_codec.encode_json(notional), Money) == notional
    unit_rows.append(
        (
            row[0],
            price.value,
            quantity.value,
            notional.value,
            source_quote,
            notional.value == source_quote,
            notional.currency.code,
        )
    )
NotebookDisplay.show_table(
    (
        "Trade ID",
        "Price",
        "Base quantity",
        "Exact product",
        "Source quote quantity",
        "Equal",
        "Currency",
    ),
    unit_rows,
)
print("Example amount with explicit currency and exact decimal text:")
print(real_codec.encode_json(notional).decode())

Trade ID,Price,Base quantity,Exact product,Source quote quantity,Equal,Currency
3620989757,67540.01000000,0.00099000,66.864609900000000000,66.86460990,True,USDT
3620989758,67540.01000000,0.00071000,47.953407100000000000,47.95340710,True,USDT
3620989759,67540.01000000,0.00118000,79.697211800000000000,79.69721180,True,USDT


Example amount with explicit currency and exact decimal text:
{"currency":{"code":"USDT","minor_unit":"0.00000001"},"value":"79.697211800000000000"}


## 7. Prove repeatability and rejection using this same object

**Input:** the actual saved manifest, ZIP and descriptor above. **Work:** reopen and verify the run twice, then test a changed copy and an injected bad publisher digest. **Output:** unchanged completion/receipt evidence and explicit failure diagnostics.

The two reopens are read-only. The failure examples use a temporary copy and a deliberately injected checksum response, make no network request, and never modify the retained archive. These controlled failures remain useful: a successful real download cannot demonstrate what a corrupted response or interrupted writer would do. The companion retains the broader process-death, retry, missing-object and clock-boundary cases.

In [9]:
# standard library imports
from http.client import HTTPSConnection
from tempfile import TemporaryDirectory
from unittest.mock import patch

# project library imports
from alpha_research.data.providers.binance.transport import BinanceArchiveTransport

before_manifest = manifest_path.read_bytes()
reopened = AcquisitionManifest.read(manifest_path, run=real_run, store=real_store)
assert reopened == real_checkpoint and manifest_path.read_bytes() == before_manifest
assert reopened.completed[0].ts_received == descriptor.ts_received
NotebookDisplay.show_table(
    ("Repeat evidence", "Observed"),
    [
        ("Manifest bytes unchanged", True),
        ("Logical completions", len(reopened.completed)),
        ("Original receipt unchanged", reopened.completed[0].ts_received),
    ],
)
real_failure_rows = []
with real_store.open(descriptor=descriptor, request=real_request) as saved:
    verified_zip = saved.read()
with TemporaryDirectory(prefix="binance-rejection-") as temporary:
    changed = Path(temporary) / "changed.zip"
    changed.write_bytes(b"X" + verified_zip[1:])
    try:
        descriptor.verify(changed, request=real_request)
    except ValueError as error:
        real_failure_rows.append(("Changed actual ZIP copy", type(error).__name__, str(error)))
    else:
        raise AssertionError("Changed source bytes were accepted")
bad_checksum = (
    b"0" * 64
    + f"  {real_request.instrument.symbol}-trades-{real_request.start.date()}.zip\n".encode()
)
with patch.object(BinanceArchiveTransport, "_get", side_effect=[bad_checksum, verified_zip]):
    try:
        BinanceArchiveTransport(
            HTTPSConnection("data.binance.vision"), timeout_seconds=10, max_bytes=BINANCE_MAX_BYTES
        ).fetch(
            real_request,
            permitted_rights_reference=real_run.job.rights_reference,
            api_key=None,
        )
    except ValueError as error:
        real_failure_rows.append(
            ("Injected wrong publisher checksum", type(error).__name__, str(error))
        )
    else:
        raise AssertionError("A mismatched publisher checksum was accepted")
NotebookDisplay.show_table(("Rejected case", "Failure", "Diagnostic"), real_failure_rows)
del verified_zip

Repeat evidence,Observed
Manifest bytes unchanged,True
Logical completions,1
Original receipt unchanged,2026-09-16 07:37:35.441268+00:00


Rejected case,Failure,Diagnostic
Changed actual ZIP copy,ValueError,"raw object SHA-256 mismatch: expected c9c333f58fe33faaa0a34d526c61b13f5648b908c4ad341dd6556e15c5bcc46a, got e958b2a046f0213ec8486b1f401468a4c395bf78669ffb354bfd76c1ea6f8300"
Injected wrong publisher checksum,ValueError,Binance archive SHA-256 differs from publisher checksum


### NB 08: equal-time order, empty content and explicit failures

These small **synthetic** saved ZIPs need no downloader. Rows with the same exchange timestamp retain descending trade IDs and exact decimal/newline bytes. A completed archive with an empty CSV yields zero rows; an absent completion and a missing stored object raise `FileNotFoundError`. Invalid UTF-8, malformed CSV and a truncated ZIP raise `ValueError`.

Every partition reports `recorder_coverage="unknown"`. The source archive supplies no recorder-gap evidence, capture clock or historical knowledge clock. Local acquisition timestamps are retained only as acquisition provenance. No case creates disconnects, padded prices or an accepted dataset.

In [10]:
# standard library imports
from io import BytesIO
from zipfile import ZIP_DEFLATED, ZipFile

# project library imports
from alpha_research.data.raw_objects import RawObject


class _MR08SavedFixture:
    """Save tiny synthetic source objects through the same installed storage boundary."""

    @staticmethod
    def save(
        directory: Path, content: bytes, *, truncated: bool = False
    ) -> tuple[BinanceArchiveRows, RawStore]:
        """Bind exact synthetic ZIP bytes to the existing expected day and research bounds."""
        with BytesIO() as output:
            with ZipFile(output, "w", compression=ZIP_DEFLATED) as archive:
                archive.writestr(
                    f"{real_request.instrument.symbol}-trades-{real_request.start.date()}.csv",
                    content,
                )
            payload = output.getvalue()
        source = directory / "synthetic.zip"
        source.write_bytes(payload[:-12] if truncated else payload)
        origin = RawObject.from_file(
            source,
            request=real_request,
            retrieval_reference="synthetic:nb08",
            ts_received=datetime(2026, 9, 16, tzinfo=UTC),
            ts_ingested=datetime(2026, 9, 16, tzinfo=UTC),
        )
        store = RawStore(directory)
        store.publish(source, descriptor=origin, request=real_request)
        manifest = AcquisitionManifest(run=real_run, completed=(origin,))
        return BinanceArchiveRows.from_manifest(manifest)[0], store


mr08_outcomes = []
with TemporaryDirectory(prefix="nb08-saved-archives-") as temporary:
    directory = Path(temporary)
    equal_time_bytes = (
        b"12,100.00,2.0,200.000,1717200000000,true,true\r\n"
        b"11,101.00,3.0,303.000,1717200000000,false,true\n"
    )
    reader, store = _MR08SavedFixture.save(directory, equal_time_bytes)
    equal_time_rows = tuple(reader.read(store, max_bytes=4096))
    assert [row.position for row in equal_time_rows] == [0, 1]
    assert [row.columns()[0] for row in equal_time_rows] == ["12", "11"]
    assert b"".join(row.payload for row in equal_time_rows) == equal_time_bytes
    NotebookDisplay.show_table(
        ("Position", "Trade ID", "Exchange time text", "Exact source bytes"),
        [
            (row.position, row.columns()[0], row.columns()[4], repr(row.payload))
            for row in equal_time_rows
        ],
    )
    NotebookDisplay.show_table(
        ("Raw origin / bounds", "Observed"),
        [
            ("ZIP SHA-256", equal_time_rows[0].origin.sha256),
            ("Request identity", equal_time_rows[0].origin.request_identity),
            ("Selection start", equal_time_rows[0].selection_start),
            ("Selection end (exclusive)", equal_time_rows[0].selection_end),
            ("Recorder coverage", reader.recorder_coverage),
        ],
    )
    empty, store = _MR08SavedFixture.save(directory, b"")
    assert tuple(empty.read(store, max_bytes=4096)) == ()
    mr08_outcomes.append(("Empty CSV", "0 rows; no completeness claim", empty.recorder_coverage))
    for label, content, truncated in (
        ("Invalid UTF-8", b"\xff,1,2,3,4,true,true\n", False),
        ("Malformed CSV", b'1,"unclosed,2,3,4,true,true\n', False),
        ("Truncated ZIP", equal_time_bytes, True),
    ):
        invalid, store = _MR08SavedFixture.save(directory, content, truncated=truncated)
        try:
            tuple(invalid.read(store, max_bytes=4096))
        except ValueError as error:
            mr08_outcomes.append((label, str(error), invalid.recorder_coverage))
        else:
            raise AssertionError(f"Accepted {label}")
    absent = BinanceArchiveRows.from_manifest(AcquisitionManifest(run=real_run))[0]
    assert empty.descriptor is not None
    (directory / empty.descriptor.sha256).unlink()
    for label, missing in (("Absent daily completion", absent), ("Missing stored ZIP", empty)):
        try:
            tuple(missing.read(store, max_bytes=4096))
        except FileNotFoundError:
            mr08_outcomes.append((label, "FileNotFoundError", missing.recorder_coverage))
        else:
            raise AssertionError(f"Accepted {label}")
NotebookDisplay.show_table(("Case", "Observed outcome", "Recorder coverage"), mr08_outcomes)

Position,Trade ID,Exchange time text,Exact source bytes
0,12,1717200000000,"b'12,100.00,2.0,200.000,1717200000000,true,true\r\n'"
1,11,1717200000000,"b'11,101.00,3.0,303.000,1717200000000,false,true\n'"


Raw origin / bounds,Observed
ZIP SHA-256,145b5173193c2fa546c4259ba1d30fd02a87c7e77f562aa4e74621279fdcd93d
Request identity,c48117ff86b3e215ac330e0568c8019242beed75968c9fdd6ada366e82452cd9
Selection start,2024-06-01 00:00:00+00:00
Selection end (exclusive),2024-06-01 01:00:00+00:00
Recorder coverage,unknown


Case,Observed outcome,Recorder coverage
Empty CSV,0 rows; no completeness claim,unknown
Invalid UTF-8,invalid Binance CSV at row 0: 'utf-8' codec can't decode byte 0xff in position 0: invalid start byte,unknown
Malformed CSV,invalid Binance CSV at row 0: unexpected end of data,unknown
Truncated ZIP,Binance archive has invalid or truncated ZIP bytes,unknown
Absent daily completion,FileNotFoundError,unknown
Missing stored ZIP,FileNotFoundError,unknown


## 8. What is connected, and what remains unknown

The same run and descriptor now connect configuration, acquisition, storage, clocks, source inspection and exact financial values. The table summarizes observed outputs from this execution. It does not infer unavailable information from a successful download.

In [11]:
NotebookDisplay.show_table(
    ("Step", "Evidence from this execution"),
    [
        ("Configured instrument", real_instrument.symbol),
        ("Stored source", f"{descriptor.size_bytes:,} bytes; SHA-256 {descriptor.sha256}"),
        ("Selected hour", f"{selected_count:,} real trades"),
        (
            "Exact financial units",
            f"{len(unit_rows)} actual trades calculated and serialization-checked",
        ),
        ("Repeat", "same manifest, one completion and original receipt"),
        ("NB 08", f"equal-time row order retained; {len(mr08_outcomes)} boundary outcomes"),
        (
            "Failure handling",
            f"{len(real_failure_rows)} explicit rejections on copies/injected input",
        ),
        (
            "Not established",
            "historical eligibility, knowledge/revision availability, complete capture coverage",
        ),
        (
            "Next package capability",
            "MR 09: bind decoded events to raw occurrences",
        ),
    ],
)

Step,Evidence from this execution
Configured instrument,BTCUSDT
Stored source,"7,308,819 bytes; SHA-256 c9c333f58fe33faaa0a34d526c61b13f5648b908c4ad341dd6556e15c5bcc46a"
Selected hour,"27,480 real trades"
Exact financial units,3 actual trades calculated and serialization-checked
Repeat,"same manifest, one completion and original receipt"
NB 08,equal-time row order retained; 6 boundary outcomes
Failure handling,2 explicit rejections on copies/injected input
Not established,"historical eligibility, knowledge/revision availability, complete capture coverage"
Next package capability,MR 09: bind decoded events to raw occurrences


For deeper explanations of capture rounding, disconnect handling, units, independent hand accounting and process interruption, open [Foundation contract checks](foundation-contract-checks.ipynb). Those examples use deliberately chosen synthetic inputs and remain separate from claims about this downloaded sample.

**Revisit a section:** keep the same resolved run when checking downstream cells. After changing inputs, restart the kernel and Run All. A different run identity uses its own manifest; do not rewrite an old manifest to make incompatible data fit. The fixed P1 hour and economic rules are unchanged.